In [ ]:
import pandas as pd
import numpy as np

from tqdm.notebook import tqdm
import os
import shutil
import argparse
import torch
from torch import nn
import torch.nn.functional as F
from torch import autograd
import torchmetrics
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torchvision
from torchvision import datasets
from torchvision.transforms import ToTensor
import torchvision.transforms.functional as TF
from torchvision.transforms import v2
from torchvision.transforms.functional import adjust_brightness, adjust_contrast

import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
from matplotlib.image import imread
import matplotlib.cm as cm
import matplotlib.image as plim
import plotly.express as px
import cv2

from tkinter import *
from tkinter import filedialog
import time
root = Tk()
root.withdraw()
root.call('wm', 'attributes', '.', '-topmost', True)
train_losses = []
test_losses = []
val_iou = []; val_acc = []
train_iou = []; train_acc = []
lrs = []
metrics = [train_losses, test_losses, val_iou, val_acc, train_iou, train_acc, lrs] 
def one_hot_encode(label, label_values):
    # label shape: (H, W, 3)
    # semantic_map shape: (NumClasses, H, W)
    semantic_map = np.zeros((len(label_values), label.shape[0], label.shape[1]))
    
    for i, colour in enumerate(label_values):
        # We check where the (H, W, 3) image matches the (3,) color vector
        # all(axis=-1) ensures all R, G, and B values match
        semantic_map[i] = np.all(label == colour, axis=-1)
        
    return semantic_map

def reverse_one_hot(image):
    """
    Transform a 2D array in one-hot format (depth is num_classes),
    to a 2D array with only 1 channel, where each pixel value is
    the classified class key.
    # Arguments
        image: The one-hot format image 
        
    # Returns
        A 2D array with the same width and hieght as the input, but
        with a depth size of 1, where each pixel value is the classified 
        class key.
    """
    x = torch.argmax(image, axis = -1)
    return x
from einops import rearrange
class ActiveLearningDataset(Dataset):
    def __init__(self, img_dir, mask_dir, weight_dir=None, numclasses=4, scalefac=1, class_assignment=None):
        valid_extensions = ('.jpg', '.jpeg', '.png', '.tif', '.tiff')
        self.img_labels = [f for f in os.listdir(img_dir) if f.lower().endswith(valid_extensions)]
        
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.weight_dir = weight_dir # NEW: directory for .npy weights
        
        self.numclasses = numclasses
        self.scalefac = scalefac
        self.class_assignment = class_assignment if class_assignment is not None else np.arange(numclasses)

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        img_name = self.img_labels[idx]
        img_path = os.path.join(self.img_dir, img_name)
        label_path = os.path.join(self.mask_dir, img_name)

        image = cv2.imread(img_path, 0)
        label = cv2.imread(label_path) # Load as BGR
        label = cv2.cvtColor(label, cv2.COLOR_BGR2RGB)

        # NEW: Load Weight Map (or default to 1s if it doesn't exist)
        if self.weight_dir is not None:
            weight_path = os.path.join(self.weight_dir, img_name + ".npy")
            if os.path.exists(weight_path):
                weight = np.load(weight_path)
            else:
                weight = np.ones((image.shape[0], image.shape[1]), dtype=np.float32)
        else:
            weight = np.ones((image.shape[0], image.shape[1]), dtype=np.float32)

        # Augmentation (Anisotropic Diffusion)
        if np.random.rand() > 0.5:
            # Note: assuming anisodiff is defined elsewhere in your code
            image = anisodiff(image, niter=np.random.randint(0, 10))

        # Convert to Tensors
        image = torch.from_numpy(image).float().unsqueeze(0) 
        label = torch.from_numpy(label).permute(2, 0, 1)    
        weight_tensor = torch.from_numpy(weight).unsqueeze(0) # NEW: (1, H, W)

        # Consistent Cropping across ALL THREE tensors
        i, j, h, w = v2.RandomCrop.get_params(image, output_size=(256 * self.scalefac, 256 * self.scalefac))
        image = TF.crop(image, i, j, h, w)
        label = TF.crop(label, i, j, h, w)
        weight_tensor = TF.crop(weight_tensor, i, j, h, w) # NEW

        # Flips
        if np.random.random() > 0.5:
            image = TF.hflip(image)
            label = TF.hflip(label)
            weight_tensor = TF.hflip(weight_tensor) # NEW
            
        if np.random.random() > 0.5:
            image = TF.vflip(image)
            label = TF.vflip(label)
            weight_tensor = TF.vflip(weight_tensor) # NEW

        # One-Hot Encoding
        label_np = label.permute(1, 2, 0).numpy() 
        encoded_label = one_hot_encode(label_np, self.class_assignment)
        label_tensor = torch.tensor(encoded_label, dtype=torch.float)
        
        return image, label_tensor, weight_tensor

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import skimage.io as io
import skimage.filters as flt
%matplotlib inline
# since we can't use imports
import numpy as np
import scipy.ndimage.filters as flt
import warnings

def anisodiff(img,niter=1,kappa=50,gamma=0.1,step=(1.,1.),sigma=0, option=1,ploton=False):
	"""
	Anisotropic diffusion.

	Usage:
	imgout = anisodiff(im, niter, kappa, gamma, option)

	Arguments:
	        img    - input image
	        niter  - number of iterations
	        kappa  - conduction coefficient 20-100 ?
	        gamma  - max value of .25 for stability
	        step   - tuple, the distance between adjacent pixels in (y,x)
	        option - 1 Perona Malik diffusion equation No 1
	                 2 Perona Malik diffusion equation No 2
	        ploton - if True, the image will be plotted on every iteration

	Returns:
	        imgout   - diffused image.

	kappa controls conduction as a function of gradient.  If kappa is low
	small intensity gradients are able to block conduction and hence diffusion
	across step edges.  A large value reduces the influence of intensity
	gradients on conduction.

	gamma controls speed of diffusion (you usually want it at a maximum of
	0.25)

	step is used to scale the gradients in case the spacing between adjacent
	pixels differs in the x and y axes

	Diffusion equation 1 favours high contrast edges over low contrast ones.
	Diffusion equation 2 favours wide regions over smaller ones.

	Reference: 
	P. Perona and J. Malik. 
	Scale-space and edge detection using ansotropic diffusion.
	IEEE Transactions on Pattern Analysis and Machine Intelligence, 
	12(7):629-639, July 1990.

	Original MATLAB code by Peter Kovesi  
	School of Computer Science & Software Engineering
	The University of Western Australia
	pk @ csse uwa edu au
	<http://www.csse.uwa.edu.au>

	Translated to Python and optimised by Alistair Muldal
	Department of Pharmacology
	University of Oxford
	<alistair.muldal@pharm.ox.ac.uk>

	June 2000  original version.       
	March 2002 corrected diffusion eqn No 2.
	July 2012 translated to Python
	"""

	# ...you could always diffuse each color channel independently if you
	# really want
	if img.ndim == 3:
		warnings.warn("Only grayscale images allowed, converting to 2D matrix")
		img = img.mean(2)

	# initialize output array
	img = img.astype('float32')
	imgout = img.copy()

	# initialize some internal variables
	deltaS = np.zeros_like(imgout)
	deltaE = deltaS.copy()
	NS = deltaS.copy()
	EW = deltaS.copy()
	gS = np.ones_like(imgout)
	gE = gS.copy()

	# create the plot figure, if requested
	if ploton:
		import pylab as pl
		from time import sleep

		fig = pl.figure(figsize=(20,5.5),num="Anisotropic diffusion")
		ax1,ax2 = fig.add_subplot(1,2,1),fig.add_subplot(1,2,2)

		ax1.imshow(img,interpolation='nearest')
		ih = ax2.imshow(imgout,interpolation='nearest',animated=True)
		ax1.set_title("Original image")
		ax2.set_title("Iteration 0")

		fig.canvas.draw()

	for ii in np.arange(1,niter):

		# calculate the diffs
		deltaS[:-1,: ] = np.diff(imgout,axis=0)
		deltaE[: ,:-1] = np.diff(imgout,axis=1)

		if 0<sigma:
			deltaSf=flt.gaussian_filter(deltaS,sigma);
			deltaEf=flt.gaussian_filter(deltaE,sigma);
		else: 
			deltaSf=deltaS;
			deltaEf=deltaE;
			
		# conduction gradients (only need to compute one per dim!)
		if option == 1:
			gS = np.exp(-(deltaSf/kappa)**2.)/step[0]
			gE = np.exp(-(deltaEf/kappa)**2.)/step[1]
		elif option == 2:
			gS = 1./(1.+(deltaSf/kappa)**2.)/step[0]
			gE = 1./(1.+(deltaEf/kappa)**2.)/step[1]

		# update matrices
		E = gE*deltaE
		S = gS*deltaS

		# subtract a copy that has been shifted 'North/West' by one
		# pixel. don't as questions. just do it. trust me.
		NS[:] = S
		EW[:] = E
		NS[1:,:] -= S[:-1,:]
		EW[:,1:] -= E[:,:-1]

		# update the image
		imgout += gamma*(NS+EW)

		if ploton:
			iterstring = "Iteration %i" %(ii+1)
			ih.set_data(imgout)
			ax2.set_title(iterstring)
			fig.canvas.draw()
			# sleep(0.01)

	return imgout

def anisodiff3(stack,niter=1,kappa=50,gamma=0.1,step=(1.,1.,1.),option=1,ploton=False):
	"""
	3D Anisotropic diffusion.

	Usage:
	stackout = anisodiff(stack, niter, kappa, gamma, option)

	Arguments:
	        stack  - input stack
	        niter  - number of iterations
	        kappa  - conduction coefficient 20-100 ?
	        gamma  - max value of .25 for stability
	        step   - tuple, the distance between adjacent pixels in (z,y,x)
	        option - 1 Perona Malik diffusion equation No 1
	                 2 Perona Malik diffusion equation No 2
	        ploton - if True, the middle z-plane will be plotted on every 
	        	 iteration

	Returns:
	        stackout   - diffused stack.

	kappa controls conduction as a function of gradient.  If kappa is low
	small intensity gradients are able to block conduction and hence diffusion
	across step edges.  A large value reduces the influence of intensity
	gradients on conduction.

	gamma controls speed of diffusion (you usually want it at a maximum of
	0.25)

	step is used to scale the gradients in case the spacing between adjacent
	pixels differs in the x,y and/or z axes

	Diffusion equation 1 favours high contrast edges over low contrast ones.
	Diffusion equation 2 favours wide regions over smaller ones.

	Reference: 
	P. Perona and J. Malik. 
	Scale-space and edge detection using ansotropic diffusion.
	IEEE Transactions on Pattern Analysis and Machine Intelligence, 
	12(7):629-639, July 1990.

	Original MATLAB code by Peter Kovesi  
	School of Computer Science & Software Engineering
	The University of Western Australia
	pk @ csse uwa edu au
	<http://www.csse.uwa.edu.au>

	Translated to Python and optimised by Alistair Muldal
	Department of Pharmacology
	University of Oxford
	<alistair.muldal@pharm.ox.ac.uk>

	June 2000  original version.       
	March 2002 corrected diffusion eqn No 2.
	July 2012 translated to Python
	"""

	# ...you could always diffuse each color channel independently if you
	# really want
	if stack.ndim == 4:
		warnings.warn("Only grayscale stacks allowed, converting to 3D matrix")
		stack = stack.mean(3)

	# initialize output array
	stack = stack.astype('float32')
	stackout = stack.copy()

	# initialize some internal variables
	deltaS = np.zeros_like(stackout)
	deltaE = deltaS.copy()
	deltaD = deltaS.copy()
	NS = deltaS.copy()
	EW = deltaS.copy()
	UD = deltaS.copy()
	gS = np.ones_like(stackout)
	gE = gS.copy()
	gD = gS.copy()

	# create the plot figure, if requested
	if ploton:
		import pylab as pl
		from time import sleep

		showplane = stack.shape[0]//2

		fig = pl.figure(figsize=(20,5.5),num="Anisotropic diffusion")
		ax1,ax2 = fig.add_subplot(1,2,1),fig.add_subplot(1,2,2)

		ax1.imshow(stack[showplane,...].squeeze(),interpolation='nearest')
		ih = ax2.imshow(stackout[showplane,...].squeeze(),interpolation='nearest',animated=True)
		ax1.set_title("Original stack (Z = %i)" %showplane)
		ax2.set_title("Iteration 0")

		fig.canvas.draw()

	for ii in np.arange(1,niter):

		# calculate the diffs
		deltaD[:-1,: ,:  ] = np.diff(stackout,axis=0)
		deltaS[:  ,:-1,: ] = np.diff(stackout,axis=1)
		deltaE[:  ,: ,:-1] = np.diff(stackout,axis=2)

		# conduction gradients (only need to compute one per dim!)
		if option == 1:
			gD = np.exp(-(deltaD/kappa)**2.)/step[0]
			gS = np.exp(-(deltaS/kappa)**2.)/step[1]
			gE = np.exp(-(deltaE/kappa)**2.)/step[2]
		elif option == 2:
			gD = 1./(1.+(deltaD/kappa)**2.)/step[0]
			gS = 1./(1.+(deltaS/kappa)**2.)/step[1]
			gE = 1./(1.+(deltaE/kappa)**2.)/step[2]

		# update matrices
		D = gD*deltaD
		E = gE*deltaE
		S = gS*deltaS

		# subtract a copy that has been shifted 'Up/North/West' by one
		# pixel. don't as questions. just do it. trust me.
		UD[:] = D
		NS[:] = S
		EW[:] = E
		UD[1:,: ,: ] -= D[:-1,:  ,:  ]
		NS[: ,1:,: ] -= S[:  ,:-1,:  ]
		EW[: ,: ,1:] -= E[:  ,:  ,:-1]

		# update the image
		stackout += gamma*(UD+NS+EW)

		if ploton:
			iterstring = "Iteration %i" %(ii+1)
			ih.set_data(stackout[showplane,...].squeeze())
			ax2.set_title(iterstring)
			fig.canvas.draw()
			# sleep(0.01)

	return stackout

In [ ]:
def pixel_accuracy(output, mask):
    with torch.no_grad():
        #print("px acc: model output: ", output.shape, "mask: ", mask.shape)
        mask = torch.argmax(mask,1).unsqueeze(1)
        output = torch.argmax(F.softmax(output, dim=1), dim=1)
        correct = torch.eq(output, mask).int()
        accuracy = float(correct.sum()) / float(correct.numel())
    return accuracy
def mIoU(pred_mask, mask, smooth=1e-10, n_classes=3):
    with torch.no_grad():
        #print("IoU: predicted: ",pred_mask.shape,"mask: ", mask.shape)
        pred_mask = F.softmax(pred_mask, dim=1)
        pred_mask = torch.argmax(pred_mask, dim=1).unsqueeze(0)
        #pred_mask = pred_mask.contiguous().view(-1)
        mask = torch.argmax(mask,1).unsqueeze(1)
        #mask = mask.contiguous().view(-1)

        iou_per_class = []
        for clas in range(0, n_classes): #loop per pixel class
            true_class = pred_mask == clas
            true_label = mask == clas

            if true_label.long().sum().item() == 0: #no exist label in this loop
                iou_per_class.append(np.nan)
            else:
                intersect = torch.logical_and(true_class, true_label).sum().float().item()
                union = torch.logical_or(true_class, true_label).sum().float().item()

                iou = (intersect + smooth) / (union +smooth)
                iou_per_class.append(iou)
        return np.nanmean(iou_per_class)
def get_lr(optimizer):
    for param_group in optimizer.param_groups:
        return param_group['lr']
testI=None
testM=None
def fit(epochs, model, train_loader, val_loader, criterion, optimizer, scheduler, patch=False,metrics=metrics,basename="modelname"):
    #torch.cuda.empty_cache()
    
    train_losses = metrics[0]
    test_losses = metrics[1]
    val_iou = metrics[2]; val_acc = metrics[3]
    train_iou = metrics[4]; train_acc = metrics[5]
    lrs = metrics[6]
    min_loss = np.inf
    decrease = 1 ; not_improve=0
    model.to(device)
    
    fit_time = time.time()
    for e in range(epochs):
        since = time.time()
        running_loss = 0
        iou_score = 0
        accuracy = 0
        #training loop
       
        model.train()
        for i, data in enumerate(tqdm(train_loader, leave=False, position=0)):
            # NEW: Unpack weights
            image_tiles, mask_tiles, weight_tiles = data 
            
            if patch:
                bs, n_tiles, c, h, w = image_tiles.size()
                image_tiles = image_tiles.view(-1, c, h, w)
                mask_tiles = mask_tiles.view(-1, h, w)
                weight_tiles = weight_tiles.view(-1, 1, h, w) # NEW
            
            image = image_tiles.to(device)
            mask = mask_tiles.to(device)
            weight = weight_tiles.to(device) # NEW
            
            output = model(image)
            
            iou_score += mIoU(output, mask)
            accuracy += pixel_accuracy(output, mask)
            
            # NEW: Pass the weight map to the criterion
            loss = criterion(output, mask, weight_map=weight) 
            
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            
            lrs.append(get_lr(optimizer))
            scheduler.step() 
            running_loss += loss.item()
            
        else:
            model.eval()
            test_loss = 0
            test_accuracy = 0
            val_iou_score = 0
            #validation loop
            with torch.no_grad():
                for i, data in enumerate(tqdm(val_loader,position=0,leave=False)):
                    #reshape to 9 patches from single image, delete batch size
                    #print(data[0].shape,data[1].shape,len(data))
                    image_tiles, mask_tiles,_ = data

                    if patch:
                        bs, n_tiles, c, h, w = image_tiles.size()

                        image_tiles = image_tiles.view(-1,c, h, w)
                        mask_tiles = mask_tiles.view(-1, h, w)
                    
                    image = image_tiles.to(device); mask = mask_tiles.to(device);
                    output = model(image)
                    #evaluation metrics
                    #print("eval")
                    val_iou_score +=  mIoU(output, mask)
                    test_accuracy += pixel_accuracy(output, mask)
                    #loss
                    loss = criterion(output, mask)                                  
                    test_loss += loss.item()
            
            #calculatio mean for each batch
            train_losses.append(running_loss/len(train_loader))
            test_losses.append(test_loss/len(val_loader))


            if min_loss > (test_loss/len(val_loader)):
                #print('Loss Decreasing.. {:.3f} >> {:.3f} '.format(min_loss, (test_loss/len(val_loader))))
                min_loss = (test_loss/len(val_loader))
                decrease += 1
                #if decrease % 5 == 0:
                    #print('saving model...')
                    #print(os.getcwd())
                    #torch.save(model, os.path.join(f'Unet-Mobilenet_v2_mIoU-focalloss-{val_iou_score/len(val_loader):.3f}-{e}.pt'))
                    

            if (test_loss/len(val_loader)) > min_loss:
                not_improve += 1
                min_loss = (test_loss/len(val_loader))
                #print(f'Loss Not Decrease for {not_improve} time')
                """if not_improve == 7:
                    #print('Loss not decrease for 7 times, Stop Training')
                    train_loader = DataLoader(train_dataset, batch_size=batchsize, shuffle=True)
                    valid_loader = DataLoader(val_dataset, batch_size=int(0.2*batchsize), shuffle=False)
                    #break"""
            
            #iou
            val_iou.append(val_iou_score/len(val_loader))
            train_iou.append(iou_score/len(train_loader))
            train_acc.append(accuracy/len(train_loader))
            val_acc.append(test_accuracy/ len(val_loader))
            print("Epoch:{}/{}..".format(e+1, epochs),
                  "Train Loss: {:.3f}..".format(running_loss/len(train_loader)),
                  "Val Loss: {:.3f}..".format(test_loss/len(val_loader)),
                  "Train mIoU:{:.3f}..".format(iou_score/len(train_loader)),
                  "Val mIoU: {:.3f}..".format(val_iou_score/len(val_loader)),
                  "Train Acc:{:.3f}..".format(accuracy/len(train_loader)),
                  "Val Acc:{:.3f}..".format(test_accuracy/len(val_loader)),
                  "Time: {:.2f}m".format((time.time()-since)/60))
        
    history = {'train_loss' : train_losses, 'val_loss': test_losses,
               'train_miou' :train_iou, 'val_miou':val_iou,
               'train_acc' :train_acc, 'val_acc':val_acc,
               'lrs': lrs}
    print('Total time: {:.2f} m' .format((time.time()- fit_time)/60))
    return history

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean',numclasses=3):
        super(FocalLoss, self).__init__()
        self.alpha = alpha # controls class imbalance
        self.gamma = gamma  # focuses on hard examples
        self.reduction = reduction
        self.numclasses = numclasses
    def forward(self, inputs, targets):
        alpha = torch.tensor([self.alpha for i in range(targets.shape[0])]).view(-1,numclasses,1,1).to(device)
        prob = F.softmax(inputs, dim=1)
        log_prob = F.log_softmax(inputs,dim=1)

        # Gather the probabilities corresponding to the correct classes


        # Apply focal adjustment
        focal_loss = -alpha* (1 - prob) ** self.gamma * torch.sum(log_prob * targets, dim=1).unsqueeze(1)

        return focal_loss.mean()
class MCDiceLoss(nn.Module):
    def __init__(self, smooth=1):
        super(MCDiceLoss, self).__init__()
        self.smooth = smooth
    def forward(self,pred, target):
        """
        Computes Dice Loss for multi-class segmentation.
        Args:
            pred: Tensor of predictions (batch_size, C, H, W).
            target: One-hot encoded ground truth (batch_size, C, H, W).
            smooth: Smoothing factor.
        Returns:
            Scalar Dice Loss.
        """
        pred = F.softmax(pred, dim=1)  # Convert logits to probabilities
        num_classes = pred.shape[1]  # Number of classes (C)
        dice = 0  # Initialize Dice loss accumulator
        
        for c in range(1,num_classes):  # Loop through each class
            pred_c = pred[:, c]  # Predictions for class c
            target_c = target[:, c]  # Ground truth for class c
            
            intersection = (pred_c * target_c).sum(dim=(1, 2))  # Element-wise multiplication
            union = pred_c.sum(dim=(1, 2)) + target_c.sum(dim=(1, 2))  # Sum of all pixels
            
            dice += (2. * intersection + self.smooth) / (union + self.smooth)  # Per-class Dice score

        return 1 - dice.mean() / num_classes  # Average Dice Loss across classes
class WeightedMCDiceLoss(nn.Module):
    def __init__(self, smooth=1):
        super(WeightedMCDiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, pred, target, weight_map=None):
        pred = F.softmax(pred, dim=1)
        num_classes = pred.shape[1]
        dice = 0
        
        for c in range(1, num_classes): # Still ignoring class 0
            pred_c = pred[:, c]
            target_c = target[:, c]
            
            if weight_map is not None:
                # Apply the spatial weight multiplier
                w = weight_map.squeeze(1)
                intersection = (pred_c * target_c * w).sum(dim=(1, 2))
                union = (pred_c * w).sum(dim=(1, 2)) + (target_c * w).sum(dim=(1, 2))
            else:
                intersection = (pred_c * target_c).sum(dim=(1, 2))
                union = pred_c.sum(dim=(1, 2)) + target_c.sum(dim=(1, 2))
                
            dice += (2. * intersection + self.smooth) / (union + self.smooth)

        return 1 - dice.mean() / num_classes

In [ ]:
import napari
import torch
import cv2
import os
import numpy as np
import scipy.ndimage as ndimage
import torch.nn.functional as F
from datetime import datetime
from scipy.signal import windows

def get_gaussian_window(size, sigma=1.0):
    """Generates a 2D Gaussian window for blending tile edges."""
    # Create a 1D Gaussian window
    window_1d = windows.gaussian(size, std=size * sigma / 2)
    # Outer product to make it 2D
    window_2d = np.outer(window_1d, window_1d)
    return torch.tensor(window_2d, dtype=torch.float32)

def predict_large_image(model, image_np, tile_size=256, overlap=0.5, device='cuda',phase_biases = None,num_classes=4):
    """
    Runs sliding-window inference with Gaussian logit blending.
    overlap: float between 0 and 1 (0.5 means 50% overlap)
    """
    if phase_biases is None:
        phase_biases = torch.zeros((num_classes,1,1), device=device)
    h, w = image_np.shape
    print(overlap,tile_size)
    stride = int(tile_size * (1 - overlap))
    
    # We will initialize these dynamically on the first tile
    full_pred = None
    count_map = None
    
    # Generate the Gaussian window and send to GPU
    window = get_gaussian_window(tile_size, sigma=0.5).to(device)
    window = window.unsqueeze(0) # Shape: (1, tile_size, tile_size)
    
    model.eval()
    with torch.no_grad():
        for y in range(0, h, stride):
            for x in range(0, w, stride):
                y1, x1 = y, x
                y2, x2 = min(y + tile_size, h), min(x + tile_size, w)
                
                tile = image_np[y1:y2, x1:x2]
                
                # Pad tile if it's smaller than tile_size (at the edges)
                pad_y = tile_size - (y2 - y1)
                pad_x = tile_size - (x2 - x1)
                if pad_y > 0 or pad_x > 0:
                    tile = np.pad(tile, ((0, pad_y), (0, pad_x)), mode='reflect')
                
                # Prep for model
                tile_tensor = torch.from_numpy(tile).float().unsqueeze(0).unsqueeze(0).to(device)
                
                # Predict (Output shape: 1, NumClasses, H, W)
                output = model(tile_tensor)
                
                # --- DYNAMIC INITIALIZATION FIX ---
                if full_pred is None:
                    num_classes = output.shape[1]
                    full_pred = torch.zeros((num_classes, h, w), device=device)
                    count_map = torch.zeros((1, h, w), device=device)
                
                # Remove batch dimension -> (NumClasses, H, W)
                output = output.squeeze(0)
                
                # Multiply logits by Gaussian window
                weighted_output = output * window
                
                # Crop the padding off if we are at the edge of the image
                weighted_output = weighted_output[:, :tile_size-pad_y, :tile_size-pad_x]
                current_window = window[:, :tile_size-pad_y, :tile_size-pad_x]
                
                # Accumulate the weighted logits and the weights
                full_pred[:, y1:y2, x1:x2] += weighted_output
                count_map[:, y1:y2, x1:x2] += current_window

    # Normalize the accumulated logits by the accumulated weights
    # Adding a small epsilon to prevent division by zero
    # Ensure phase_biases is (C, 1, 1) to broadcast over (C, H, W)
    # Let's say we want to boost Phase 1 (index 1)
    #phase_biases = torch.zeros((4, 1, 1), device=device)
    #phase_biases[1, 0, 0] = 2.0  # Start with 2.0 and adjust based on logit range

    # Apply after normalization
    full_pred_normalized = full_pred / (count_map + 1e-8)
    full_pred_biased = full_pred_normalized + phase_biases
    print(phase_biases.shape)
    # Argmax
    final_mask = torch.argmax(F.softmax(full_pred_biased, dim=0), dim=0)
    #full_pred = phase_biases+full_pred / (count_map + 1e-8)
    #
    ## Apply Softmax and Argmax
    #final_mask = torch.argmax(F.softmax(full_pred, dim=0), dim=0)
    
    return final_mask.cpu().numpy().astype(np.uint8)
def map_to_grayscale(mask_np, uniques=np.array([0, 85, 170, 255])):
    """Maps class indices (0,1,2,3) to grayscale intensity values."""
    return uniques[mask_np].astype(np.uint8)

def review_full_cross_section(model, img_path, save_img_dir, save_mask_dir, tile_size=256, device='cuda'):
    """
    Loads a large image, stitches the tiled predictions, and opens Napari.
    """
    os.makedirs(save_img_dir, exist_ok=True)
    os.makedirs(save_mask_dir, exist_ok=True)
    
    img_name = os.path.basename(img_path)
    
    # Load full large image
    image_np = cv2.imread(img_path, 0)
    if image_np is None:
        raise FileNotFoundError(f"Could not load {img_path}")

    # Get stitched prediction
    print("Running sliding window inference...")
    full_mask_np = predict_large_image(model, image_np, tile_size=tile_size, device=device)
    
    # Launch Napari
    viewer = napari.Viewer()
    viewer.add_image(image_np, name='Full SEM Cross-Section', colormap='gray')
    labels_layer = viewer.add_labels(full_mask_np, name='Editable Mask')

    @viewer.bind_key('s')
    def save_approved_data(viewer):
        approved_mask = labels_layer.data.astype(np.uint8)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        out_name = f"refined_full_{timestamp}_{img_name}"
        
        # Save large image and mask
        cv2.imwrite(os.path.join(save_img_dir, out_name), image_np)
        
        # Save mask as grayscale integers (0, 1, 2). 
        # Note: If your __getitem__ needs an RGB mask, we must map these integers to colors first!
        cv2.imwrite(os.path.join(save_mask_dir, out_name), approved_mask)
        
        print(f"Saved full refined data: {out_name}")
        viewer.status = f"Saved: {out_name}"
        
    napari.run()

def review_batch_stepwise(model, image_paths, save_img_dir, save_mask_dir, save_weight_dir, tile_size=256, device='cuda'):
    """
    Processes a small batch of images sequentially. 
    Calculates error maps based on user corrections.
    """
    os.makedirs(save_img_dir, exist_ok=True)
    os.makedirs(save_mask_dir, exist_ok=True)
    os.makedirs(save_weight_dir, exist_ok=True) # Directory for the weight maps

    for img_path in image_paths:
        print(f"Loading {img_path}...")
        image_np = cv2.imread(img_path, 0)
        img_name = os.path.basename(img_path)
        
        # 1. Predict (Using the Gaussian sliding window from earlier)
        original_mask_np = predict_large_image(model, image_np, tile_size, overlap=0.5, device=device)
        
        # 2. Open Napari for this single image
        viewer = napari.Viewer()
        viewer.add_image(image_np, name='SEM Image', colormap='gray')
        labels_layer = viewer.add_labels(original_mask_np, name='Editable Mask')

        @viewer.bind_key('s')
        def save_approved_data(viewer, current_img=img_name, orig_mask=original_mask_np, img_data=image_np):
            edited_mask = labels_layer.data.astype(np.uint8)
            
            # --- CALCULATE ERROR WEIGHT MAP ---
            # Base weight is 1.0. If the pixel was changed by the user, weight is 10.0
            weight_map = np.where(orig_mask != edited_mask, 10.0, 1.0).astype(np.float32)
            
            # --- MAP TO GRAYSCALE (0, 85, 170, 255) ---
            mapped_mask = map_to_grayscale(edited_mask)
            
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            base_name = f"refined_{timestamp}_{current_img}"
            
            # Save Image, Mask, and Weight Map
            cv2.imwrite(os.path.join(save_img_dir, base_name), img_data)
            cv2.imwrite(os.path.join(save_mask_dir, base_name), mapped_mask)
            np.save(os.path.join(save_weight_dir, base_name + ".npy"), weight_map)
            
            print(f"Saved: {base_name} (with error weights)")
            viewer.close() # Auto-close to move to the next image

        print(f"Reviewing {img_name}. Press 's' to save and proceed to the next.")
        napari.run()
def review_and_autocrop(model, img_path, save_img_dir, save_mask_dir, save_weight_dir, tile_size=256, device='cuda'):
    """
    Opens the full image. Upon saving, automatically finds user edits, 
    crops a tile_size x tile_size patch around each distinct edit, and saves them.
    """
    os.makedirs(save_img_dir, exist_ok=True)
    os.makedirs(save_mask_dir, exist_ok=True)
    os.makedirs(save_weight_dir, exist_ok=True)

    img_name = os.path.basename(img_path)
    image_np = cv2.imread(img_path, 0)
    h, w = image_np.shape

    # 1. Predict full image (assuming predict_large_image is defined from earlier)
    print("Generating full segmentation...")
    original_mask_np = predict_large_image(model, image_np, tile_size=256, overlap=0.5, device=device)
    print("vol fracs: ",[np.mean(original_mask_np==i) for i in np.unique(original_mask_np)])
    # 2. Open Napari
    viewer = napari.Viewer()
    viewer.add_image(image_np, name='SEM Image', colormap='gray')
    labels_layer = viewer.add_labels(original_mask_np.copy(), name='Editable Mask')

    @viewer.bind_key('s')
    def save_autocropped_data(viewer):
        edited_mask = labels_layer.data.astype(np.uint8)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        
        # Find where the user made changes
        diff_mask = (original_mask_np != edited_mask)
        
        if not np.any(diff_mask):
            print("No edits detected. Closing.")
            viewer.close()
            return
            
        # Group edited pixels into distinct clusters (features)
        labeled_diff, num_features = ndimage.label(diff_mask)
        print(f"Detected {num_features} distinct edit regions. Cropping patches...")

        saved_count = 0
        for i in range(1, num_features + 1):
            # Find the center (centroid) of the edit
            coords = np.argwhere(labeled_diff == i)
            y_c, x_c = coords.mean(axis=0).astype(int)
            
            # Calculate 256x256 crop boundaries, keeping them within the image limits
            y1 = max(0, y_c - tile_size // 2)
            y2 = min(h, y1 + tile_size)
            x1 = max(0, x_c - tile_size // 2)
            x2 = min(w, x1 + tile_size)
            
            # Adjust if we hit the right/bottom edges
            if y2 - y1 < tile_size: y1 = max(0, y2 - tile_size)
            if x2 - x1 < tile_size: x1 = max(0, x2 - tile_size)

            # Crop the data
            img_patch = image_np[y1:y2, x1:x2]
            mask_patch = edited_mask[y1:y2, x1:x2]
            orig_mask_patch = original_mask_np[y1:y2, x1:x2]
            
            # Create weight map: 10.0 for changed pixels, 1.0 for untouched
            weight_patch = np.where(mask_patch != orig_mask_patch, 10.0, 1.0).astype(np.float32)
            
            # Save
            patch_name = f"autocrop_{y_c}_{x_c}_{timestamp}_{img_name}"
            cv2.imwrite(os.path.join(save_img_dir, patch_name), img_patch)
            cv2.imwrite(os.path.join(save_mask_dir, patch_name), map_to_grayscale(mask_patch))
            np.save(os.path.join(save_weight_dir, patch_name + ".npy"), weight_patch)
            
            saved_count += 1
            
        print(f"Saved {saved_count} active learning patches.")
        viewer.close()

    print("Reviewing full image. Make your edits and press 's' to auto-crop.")
    napari.run()

In [ ]:
import torch
import torch.nn.functional as F
import napari
import cv2
import os
import json
import random
from datetime import datetime
import numpy as np
def get_shuffled_grid_tiles(h, w, tile_size, registry, max_tiles=5):
    """
    Creates a full grid of tiles, shuffles them, and returns the next 
    available unapproved ones.
    """
    all_coords = []
    for y in range(0, h - tile_size + 1, tile_size):
        for x in range(0, w - tile_size + 1, tile_size):
            all_coords.append((y, x))
    
    # Shuffle so we see different parts of the pellet in each batch
    random.shuffle(all_coords)
    
    active_tiles = []
    for (y, x) in all_coords:
        coord_key = f"{y}_{x}"
        if coord_key not in registry:
            active_tiles.append((y, x))
        
        if len(active_tiles) >= max_tiles:
            break
            
    return active_tiles


def review_large_image_shuffled(model, img_path, save_img_dir, save_mask_dir, save_weight_dir, 
                                tile_size=256, max_tiles_per_step=5, device='cuda', 
                                model_weights_path=None, reedit_mode=False,img=None):
    
    # 1. Force Absolute Paths (Prevents saving in wrong locations after kernel restart)
    save_img_dir = os.path.abspath(save_img_dir)
    save_mask_dir = os.path.abspath(save_mask_dir)
    save_weight_dir = os.path.abspath(save_weight_dir)
    for d in [save_img_dir, save_mask_dir, save_weight_dir]:
        os.makedirs(d, exist_ok=True)

    # 2. Setup Files
    img_name = os.path.basename(img_path)
    base_name = os.path.splitext(img_name)[0]
    registry_path = os.path.abspath(f"{base_name}_shuffled_registry.json")
    master_mask_path = os.path.abspath(f"{base_name}_master_mask.png")
    if img is None:
        image_np = cv2.imread(img_path, 0)
    else: image_np = img
    h, w = image_np.shape

    # 3. Load Registry Safely
    registry = {}
    if os.path.exists(registry_path) and os.path.getsize(registry_path) > 0:
        with open(registry_path, 'r') as f:
            registry = json.load(f)

    # 4. Master Mask with Explicit Mapping
    uniques = np.array([0, 85, 170, 255])
    if os.path.exists(master_mask_path):
        master_mask_gray = cv2.imread(master_mask_path, 0)
        master_mask_np = np.zeros((h, w), dtype=np.uint8)
        for idx, val in enumerate(uniques):
            master_mask_np[master_mask_gray == val] = idx
    else:
        master_mask_np = np.zeros((h, w), dtype=np.uint8)

    # 5. Tile Selection
    if reedit_mode:
        approved_coords = [tuple(map(int, k.split('_'))) for k in registry.keys()]
        if not approved_coords: return print("No tiles to re-edit!")
        active_tiles = random.sample(approved_coords, min(len(approved_coords), max_tiles_per_step))
    else:
        active_tiles = get_shuffled_grid_tiles(h, w, tile_size, registry, max_tiles_per_step)

    if not active_tiles: return print("No active tiles.")

    # 6. Predictions
    model.eval()
    orig_preds = {}
    rects = []
    with torch.no_grad():
        for (y, x) in active_tiles:
            y2, x2 = y + tile_size, x + tile_size
            tile_t = torch.from_numpy(image_np[y:y2, x:x2]).float().unsqueeze(0).unsqueeze(0).to(device)
            pred = torch.argmax(F.softmax(model(tile_t), dim=1), dim=1).squeeze().cpu().numpy()
            if not reedit_mode:
                master_mask_np[y:y2, x:x2] = pred
            orig_preds[(y, x)] = pred.copy()
            rects.append([[y, x], [y, x2], [y2, x2], [y2, x]])

    # 7. Napari Launch
    try: napari.viewer.close_all()
    except: pass
    
    viewer = napari.Viewer()
    viewer.add_image(image_np, name='SEM', colormap='gray')
    labels_layer = viewer.add_labels(master_mask_np, name='Segmentation')
    viewer.add_shapes(rects, shape_type='polygon', edge_color='yellow', face_color='transparent', name='Active')

    # 8. THE SAVE CALLBACK (Fixed closures)
    @viewer.bind_key('s')
    def save_callback(v):
        # We grab data directly from the layer to ensure we have the LATEST edits
        current_mask = labels_layer.data.astype(np.uint8)
        ts = datetime.now().strftime("%H%M%S")
        
        for (y, x) in active_tiles:
            y2, x2 = y + tile_size, x + tile_size
            refined = current_mask[y:y2, x:x2]
            
            # Save RETRAINING data
            weight_map = np.where(orig_preds[(y, x)] != refined, 15.0, 1.0).astype(np.float32)
            p_id = f"patch_{y}_{x}_{ts}"
            
            cv2.imwrite(os.path.join(save_img_dir, f"{p_id}.png"), image_np[y:y2, x:x2])
            cv2.imwrite(os.path.join(save_mask_dir, f"{p_id}.png"), uniques[refined])
            np.save(os.path.join(save_weight_dir, f"{p_id}.npy"), weight_map)
            
            registry[f"{y}_{x}"] = True

        # Save PROGRESS files
        cv2.imwrite(master_mask_path, uniques[current_mask])
        with open(registry_path, 'w') as f:
            json.dump(registry, f)
            
        print(f"--- SUCCESS: Saved {len(active_tiles)} tiles to {save_mask_dir} ---")
        v.close()

    napari.run()

In [ ]:
from backbones_unet.model.unet import Unet
from backbones_unet.utils.dataset import SemanticSegmentationDataset
from backbones_unet.model.losses import DiceLoss
from backbones_unet.utils.trainer import Trainer

# create a torch.utils.data.Dataset/DataLoader

# Get train and val data loaders
# uniques in the dataset for testing are [  0, 128, 255] for unlabelled, pores, CAM, SE

batchsize = 8
scalefac=1


model = Unet(
    backbone='resnet34', #'convnext_base', # backbone network name
    in_channels=1,            # input channels (1 for gray-scale images, 3 for RGB, etc.)
    num_classes=2,            # output channels (number of classes in your dataset)
)

# Get UNet model
model = torch.load("Unet-binaryporespace.pt")
model.to("cuda")

In [ ]:
#root = filedialog.askdirectory()

project_name = "FOLDER"
train_img_path = project_name+"/imgs"
train_mask_path = project_name+"/masks"
train_weight_path = project_name+"/weights"
#os.makedirs(project_name+"/models")
# take any validation dataset that is suitable
root = filedialog.askdirectory()
val_img_path = root+ "/images"
val_mask_path = root+"/masks"


In [ ]:
img = cv2.imread(img_path,0)
imgm = anisodiff(img,niter=5)
plt.imshow(imgm[:100,:100],cmap="gray")

In [ ]:
# Active learning workflow - check single tiles and adjust segmentations
# either just load the image or provide a modified image
review_large_image_shuffled(model, img_path, f"{project_name}/imgs", f"{project_name}/masks", f"{project_name}/weights", 
                            tile_size=256, max_tiles_per_step=1,reedit_mode=False,img=imgm)

In [ ]:
immod = imgm#anisodiff(im,niter=1)#anisodiff(im*1.1,niter=10)#im[200:-200,10:-20]# anisodiff(im,niter=10,gamma=0.2)
device="cuda"
# To make class 1 (e.g., value 85) expand, add a bias of 1.0 or 2.0
phase_biases = torch.tensor([0.0, 0.80], device=device).view(1, 2, 1, 1)

#immod[immod>230]=255
x,y = np.array(immod.shape)//256
r1,r2 = np.array(immod.shape)%256

imneu = np.zeros(immod.shape)
for i in range(x):
    for j in range(y):
        test = immod[256*i:(i+1)*256,j*256:(j+1)*256]
        #print(test.shape)
        output = F.softmax(model(torch.tensor(test,dtype=torch.float).unsqueeze(0).unsqueeze(0).to(device)),1)
        output = torch.argmax(output+phase_biases,1).cpu().numpy()
        imneu[256*i:(i+1)*256,j*256:(j+1)*256] = output
        
for i in range(x):
    test = immod[256*i:(i+1)*256,-256:]
    output = torch.argmax(F.softmax(model(torch.tensor(test,dtype=torch.float).unsqueeze(0).unsqueeze(0).to(device)),1),1).cpu().numpy()
    imneu[256*i:(i+1)*256,-256:] = output
for j in range(y):
    test = immod[-256:,j*256:(j+1)*256]
    output = torch.argmax(F.softmax(model(torch.tensor(test,dtype=torch.float).unsqueeze(0).unsqueeze(0).to(device)),1),1).cpu().numpy()
    imneu[-256:,j*256:(j+1)*256] = output
test = immod[-256:,-256:]
output = torch.argmax(F.softmax(model(torch.tensor(test,dtype=torch.float).unsqueeze(0).unsqueeze(0).to(device)),1),1).cpu().numpy()
imneu[-256:,-256:] = output

vf = [np.mean(imneu==i) for i in np.unique(imneu)]
print("vol fracs: ",vf)

In [ ]:
# save the image if you are happy with the results
plt.imsave(img_path[:-4]+"_segmented.png",imneu)#,cmap="gray",vmin=0,vmax=255)

In [ ]:
# adjust the model by training
# Once you have saved ~50 patches, run this to fine-tune the model.
# Point your ActiveLearningDataset to the "051/..." folders.
batchsize=1
train_dataset = ActiveLearningDataset(train_img_path, train_mask_path,train_weight_path,class_assignment=uniques,scalefac=scalefac)
val_dataset = ActiveLearningDataset(val_img_path, val_mask_path,weight_dir=None,class_assignment=uniques,scalefac=scalefac)
train_loader = DataLoader(train_dataset, batch_size=batchsize, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=int(batchsize), shuffle=False)
max_lr = 1e-3
epoch = 10
weight_decay = 1e-4
device="cuda"
#criterion = FocalLoss([0,0.5,0.4,0.2],2)
criterion = WeightedMCDiceLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=max_lr, weight_decay=weight_decay)
sched = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr, epochs=epoch,
                                            steps_per_epoch=len(train_loader))
history = fit(epoch, model, train_loader, val_loader, criterion, optimizer, sched)

# Save weights to disk for the next annotation session
#